# 🐶 Dog Breed Classification with Transfer Learning (PyTorch)

End-to-end image classification project: from raw images to a deployable model that powers a web app.

**Dataset:** [Dog Breed Image Dataset (Kaggle)](https://www.kaggle.com/datasets/khushikhushikhushi/dog-breed-image-dataset) — ~1,000 images across 10 breeds
(Beagle, Boxer, Bulldog, Dachshund, German Shepherd, Golden Retriever, Labrador Retriever, Poodle, Rottweiler, Yorkshire Terrier).

**Approach**
1. Explore the data (class balance, image sizes, samples)
2. Stratified train / validation / test split (70 / 15 / 15)
3. Data augmentation to fight overfitting on a small dataset
4. Transfer learning with **EfficientNet-B0** pretrained on ImageNet
   - Stage 1: train only the new classifier head (backbone frozen)
   - Stage 2: fine-tune the whole network with a low learning rate + cosine schedule
5. Evaluate on the held-out test set (accuracy, per-class report, confusion matrix, error analysis)
6. Export the weights + class names for the Streamlit web app

> **Where to run:** this notebook is designed for a **Kaggle Notebook** (free GPU). Click *Add Input* → search “dog breed image dataset” by *khushikhushikhushi* → turn on **GPU T4** under *Settings → Accelerator*. It also runs on Google Colab (it downloads the data with `kagglehub`) or locally.

## 1. Setup

In [ ]:
import os, json, random, time, copy, shutil, subprocess, sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

### Configuration

In [ ]:
# Paths -----------------------------------------------------------------------
# Leave DATA_DIR as None to auto-detect (Kaggle input folder, or download via kagglehub).
DATA_DIR = os.environ.get("DATA_DIR")  # e.g. "/path/to/dog-breed-image-dataset"
IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path(".")
MODEL_DIR = WORK_DIR / "model"     # weights + class names (used by streamlit_app.py)
ASSET_DIR = WORK_DIR / "assets"    # figures for the README
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)

# Training hyper-parameters -----------------------------------------------------
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_HEAD = int(os.environ.get("EPOCHS_HEAD", 5))     # stage 1: frozen backbone
EPOCHS_FINETUNE = int(os.environ.get("EPOCHS_FT", 12))  # stage 2: full fine-tune
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
PATIENCE = 4                      # early stopping on validation accuracy
PRETRAINED = os.environ.get("PRETRAINED", "1") == "1"
NUM_WORKERS = int(os.environ.get("NUM_WORKERS", 2))
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

## 2. Get the data

In [ ]:
KAGGLE_DATASET = "khushikhushikhushi/dog-breed-image-dataset"


def find_dataset_root(base):
    # Return the shallowest folder whose sub-folders contain images (one sub-folder per class).
    # os.walk(followlinks=True) is used because Kaggle mounts input datasets as symlinks.
    base = Path(base)
    if not base.exists():
        return None
    dirs = [base]
    for root, subdirs, _ in os.walk(base, followlinks=True):
        dirs += [Path(root) / s for s in subdirs]
    for d in sorted(dirs, key=lambda p: len(p.parts)):
        try:
            class_dirs = [s for s in d.iterdir() if s.is_dir()
                          and any(f.suffix.lower() in IMG_EXTS for f in s.iterdir() if f.is_file())]
        except (PermissionError, OSError):
            continue
        if len(class_dirs) >= 2:
            return d
    return None


def download_with_kagglehub():
    try:
        import kagglehub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
    return kagglehub.dataset_download(KAGGLE_DATASET)


DATA_ROOT = None
if DATA_DIR:                                   # 1) explicit path
    DATA_ROOT = find_dataset_root(DATA_DIR)
if DATA_ROOT is None and Path("/kaggle/input").exists():   # 2) attached via "Add Input"
    DATA_ROOT = find_dataset_root("/kaggle/input")
    if DATA_ROOT is None:
        print("Dataset not found in /kaggle/input. Contents:",
              [str(p) for p in Path("/kaggle/input").glob("*/*")][:20])
if DATA_ROOT is None:                          # 3) download (Colab / local / Kaggle with Internet on)
    print("Downloading the dataset with kagglehub ...")
    try:
        DATA_ROOT = find_dataset_root(download_with_kagglehub())
    except Exception as e:
        print("kagglehub download failed:", e)

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find the dog breed images.\n"
        "On Kaggle: click 'Add Input' (right panel) -> search 'Dog Breed Image Dataset' by "
        "khushikhushikhushi -> add it, OR turn on Settings -> Internet so it can be downloaded.\n"
        "Elsewhere: set DATA_DIR to the folder that contains the breed sub-folders.")
print("Dataset root:", DATA_ROOT)

In [ ]:
def pretty(name):
    # 'german_shepherd' / 'German-Shepherd' -> 'German Shepherd'
    return " ".join(w.capitalize() for w in name.replace("_", " ").replace("-", " ").split())

raw_classes = sorted(d.name for d in DATA_ROOT.iterdir() if d.is_dir())
records, bad_files = [], []
for label, cls in enumerate(raw_classes):
    for f in sorted((DATA_ROOT / cls).iterdir()):
        if f.suffix.lower() not in IMG_EXTS:
            continue
        try:
            with Image.open(f) as im:
                im.verify()          # skip corrupt / truncated files
            records.append({"path": str(f), "label": label, "breed": pretty(cls)})
        except Exception:
            bad_files.append(str(f))

df = pd.DataFrame(records)
CLASS_NAMES = [pretty(c) for c in raw_classes]
NUM_CLASSES = len(CLASS_NAMES)
print(f"{len(df)} valid images | {NUM_CLASSES} classes | {len(bad_files)} corrupt files skipped")
print(CLASS_NAMES)

## 3. Exploratory data analysis

In [ ]:
counts = df["breed"].value_counts().reindex(CLASS_NAMES)
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=counts.values, y=counts.index, ax=ax, color="#4C72B0")
ax.set_xlabel("Number of images"); ax.set_ylabel(""); ax.set_title("Images per breed")
for i, v in enumerate(counts.values):
    ax.text(v + 1, i, str(v), va="center")
plt.tight_layout(); plt.savefig(ASSET_DIR / "class_distribution.png", dpi=120); plt.show()

In [ ]:
# Image dimensions (sampled) — tells us how much resizing will happen
sample = df.sample(min(300, len(df)), random_state=SEED)
sizes = []
for p in sample["path"]:
    with Image.open(p) as im:
        sizes.append((im.width, im.height, im.mode))
sizes = pd.DataFrame(sizes, columns=["width", "height", "mode"])
print(sizes[["width", "height"]].describe().round(0))
print("\nColour modes:", sizes["mode"].value_counts().to_dict())

In [ ]:
# A few example images per breed
n_per_class = 4
fig, axes = plt.subplots(NUM_CLASSES, n_per_class, figsize=(n_per_class * 2.6, NUM_CLASSES * 2.6))
for r, breed in enumerate(CLASS_NAMES):
    paths = df[df["breed"] == breed]["path"].sample(min(n_per_class, (df["breed"] == breed).sum()),
                                                    random_state=SEED).tolist()
    for c in range(n_per_class):
        ax = axes[r, c] if NUM_CLASSES > 1 else axes[c]
        ax.axis("off")
        if c < len(paths):
            ax.imshow(Image.open(paths[c]).convert("RGB"))
        if c == 0:
            ax.set_title(breed, loc="left", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig(ASSET_DIR / "sample_images.png", dpi=90); plt.show()

## 4. Train / validation / test split
Stratified so every breed is represented in the same proportion in each split.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)
print(f"train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")

## 5. Datasets, augmentation and data loaders

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet statistics

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([           # identical to model_utils.get_eval_transform()
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class DogDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]


loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
train_loader = DataLoader(DogDataset(train_df, train_tf), shuffle=True, **loader_kw)
val_loader = DataLoader(DogDataset(val_df, eval_tf), shuffle=False, **loader_kw)
test_loader = DataLoader(DogDataset(test_df, eval_tf), shuffle=False, **loader_kw)

In [ ]:
# Visualise one augmented batch
def denorm(t):
    t = t.permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)
    return np.clip(t, 0, 1)

xb, yb = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, img, y in zip(axes.flat, xb, yb):
    ax.imshow(denorm(img)); ax.set_title(CLASS_NAMES[y], fontsize=9); ax.axis("off")
plt.suptitle("Augmented training samples"); plt.tight_layout(); plt.show()

## 6. Model — EfficientNet-B0 with a new head

In [ ]:
def build_model(num_classes, pretrained=True):
    # Must match model_utils.build_model so the app can load the weights
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_features, num_classes))
    return model

model = build_model(NUM_CLASSES, PRETRAINED).to(device)
total = sum(p.numel() for p in model.parameters())
print(f"EfficientNet-B0 | {total/1e6:.2f}M parameters | pretrained={PRETRAINED}")

## 7. Training

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n


history = {"epoch": [], "stage": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best = {"acc": -1.0, "loss": float("inf"), "state": None, "epoch": 0}


def fit(model, optimizer, epochs, stage, scheduler=None):
    stale = 0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader)
        if scheduler: scheduler.step()
        e = len(history["epoch"]) + 1
        for k, v in zip(history, [e, stage, tr_loss, tr_acc, va_loss, va_acc]):
            history[k].append(v)
        improved = va_acc > best["acc"] or (va_acc == best["acc"] and va_loss < best["loss"])
        if improved:
            best.update(acc=va_acc, loss=va_loss, state=copy.deepcopy(model.state_dict()), epoch=e)
            stale = 0
        else:
            stale += 1
        print(f"[{stage}] epoch {e:2d} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
              f"val loss {va_loss:.3f} acc {va_acc:.3f} | {time.time()-t0:.0f}s {'*' if improved else ''}")
        if stage == "finetune" and stale >= PATIENCE:
            print(f"Early stopping: no val improvement for {PATIENCE} epochs.")
            break

### Stage 1 — train the classifier head only

In [ ]:
for p in model.features.parameters():
    p.requires_grad = False
opt_head = torch.optim.AdamW(model.classifier.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
fit(model, opt_head, EPOCHS_HEAD, stage="head")

### Stage 2 — fine-tune the whole network

In [ ]:
for p in model.parameters():
    p.requires_grad = True
opt_ft = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=max(EPOCHS_FINETUNE, 1))
fit(model, opt_ft, EPOCHS_FINETUNE, stage="finetune", scheduler=sched)

model.load_state_dict(best["state"])
print(f"\nBest validation accuracy {best['acc']:.3f} at epoch {best['epoch']}")

In [ ]:
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric in zip(axes, ["loss", "acc"]):
    ax.plot(hist["epoch"], hist[f"train_{metric}"], "o-", label="train")
    ax.plot(hist["epoch"], hist[f"val_{metric}"], "o-", label="validation")
    if EPOCHS_HEAD and len(hist) > EPOCHS_HEAD:
        ax.axvline(EPOCHS_HEAD + 0.5, ls="--", c="gray")
        ax.text(EPOCHS_HEAD + 0.6, ax.get_ylim()[1], " fine-tune →", va="top", color="gray")
    ax.set_xlabel("epoch"); ax.set_title("Loss" if metric == "loss" else "Accuracy"); ax.legend()
plt.tight_layout(); plt.savefig(ASSET_DIR / "training_curves.png", dpi=120); plt.show()

## 8. Evaluation on the held-out test set

In [ ]:
@torch.inference_mode()
def predict_loader(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        probs.append(torch.softmax(model(x.to(device)), 1).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

test_probs, y_true = predict_loader(model, test_loader)
y_pred = test_probs.argmax(1)
test_acc = accuracy_score(y_true, y_pred)
top3 = np.mean([t in np.argsort(p)[-3:] for p, t in zip(test_probs, y_true)])
print(f"Test accuracy: {test_acc:.4f} | Top-3 accuracy: {top3:.4f}\n")
print(classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)),
                            target_names=CLASS_NAMES, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues", cbar_kws={"label": "recall"},
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion matrix — test accuracy {test_acc:.1%}")
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
plt.savefig(ASSET_DIR / "confusion_matrix.png", dpi=120); plt.show()

### Error analysis — what does the model get wrong?

In [ ]:
wrong = np.where(y_pred != y_true)[0]
print(f"{len(wrong)} misclassified out of {len(y_true)} test images")
if len(wrong):
    show = wrong[:12]
    cols = min(6, len(show)); rows = int(np.ceil(len(show) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.8, rows * 3.1), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, i in zip(axes.flat, show):
        ax.imshow(Image.open(test_df.iloc[i]["path"]).convert("RGB"))
        ax.set_title(f"true: {CLASS_NAMES[y_true[i]]}\npred: {CLASS_NAMES[y_pred[i]]} ({test_probs[i].max():.0%})",
                     fontsize=8, color="crimson")
    plt.tight_layout(); plt.show()

## 9. Predict a single image
The same function logic is used by the web app.

In [ ]:
@torch.inference_mode()
def predict_image(path_or_pil, top_k=3):
    img = Image.open(path_or_pil) if isinstance(path_or_pil, (str, Path)) else path_or_pil
    x = eval_tf(img.convert("RGB")).unsqueeze(0).to(device)
    p = torch.softmax(model.eval()(x), 1)[0].cpu()
    v, i = p.topk(min(top_k, NUM_CLASSES))
    return [(CLASS_NAMES[j], float(s)) for s, j in zip(v, i)]

fig, axes = plt.subplots(1, 4, figsize=(15, 4.5))
for ax, (_, row) in zip(axes, test_df.sample(4, random_state=1).iterrows()):
    preds = predict_image(row["path"])
    ax.imshow(Image.open(row["path"]).convert("RGB")); ax.axis("off")
    ax.set_title(f"true: {row['breed']}\n" + "\n".join(f"{n}: {s:.0%}" for n, s in preds), fontsize=9)
plt.tight_layout(); plt.show()

## 10. Export the model for the web app

In [ ]:
WEIGHTS_FILE = MODEL_DIR / "dog_breed_efficientnet_b0.pth"
torch.save(model.state_dict(), WEIGHTS_FILE)

with open(MODEL_DIR / "class_names.json", "w") as f:
    json.dump({"classes": CLASS_NAMES}, f, indent=2)

metrics = {
    "architecture": "EfficientNet-B0 (ImageNet pretrained, fine-tuned)",
    "num_classes": NUM_CLASSES,
    "train_images": len(train_df), "val_images": len(val_df), "test_images": len(test_df),
    "best_val_accuracy": round(float(best["acc"]), 4),
    "test_accuracy": round(float(test_acc), 4),
    "test_top3_accuracy": round(float(top3), 4),
    "epochs_trained": len(history["epoch"]),
}
with open(MODEL_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Sanity check: reload from disk and confirm predictions match
reloaded = build_model(NUM_CLASSES, pretrained=False)
reloaded.load_state_dict(torch.load(WEIGHTS_FILE, map_location="cpu", weights_only=True))
reloaded.eval()
x, _ = next(iter(test_loader))
with torch.inference_mode():
    assert torch.equal(reloaded(x[:4]).argmax(1), model.cpu().eval()(x[:4]).argmax(1))
model.to(device)

zip_path = shutil.make_archive(str(WORK_DIR / "dog_breed_artifacts"), "zip", WORK_DIR, "model")
shutil.make_archive(str(WORK_DIR / "dog_breed_assets"), "zip", WORK_DIR, "assets")
print(json.dumps(metrics, indent=2))
print(f"\nSaved weights ({WEIGHTS_FILE.stat().st_size/1e6:.1f} MB) ->", WEIGHTS_FILE)
print("Download:", zip_path, "and dog_breed_assets.zip")

In [ ]:
# On Google Colab, trigger the downloads automatically (no-op elsewhere)
try:
    from google.colab import files
    files.download(str(WORK_DIR / "dog_breed_artifacts.zip"))
    files.download(str(WORK_DIR / "dog_breed_assets.zip"))
except ImportError:
    print("Not on Colab — on Kaggle, open the 'Output' panel (right side) to download the zip files.")

## 11. Next steps

1. Unzip `dog_breed_artifacts.zip` into the repo's `model/` folder and `dog_breed_assets.zip` into `assets/`.
2. Run the app locally: `pip install -r requirements.txt && streamlit run streamlit_app.py`
3. Deploy to Streamlit Community Cloud (see the README) to get a public link.

**Ideas to improve:** try larger backbones (EfficientNet-B3, ConvNeXt-Tiny), test-time augmentation, MixUp/CutMix, or add a "not a dog" detector so the app rejects non-dog photos.